# 02 · Feature Aggregation (memory-aware)

Depth-1 and depth-2 tables hold many rows per application. Each table is aggregated to one row per `case_id`,
**chunk by chunk with the streaming engine**, and written to disk immediately (`features/`), so peak memory stays
low even for the ~190M-row `credit_bureau_a_2`.

**Time handling (important for stability):**
- Date columns (suffix `D`) → days relative to the decision date (negative = before the decision).
- Integer calendar-year columns (e.g. `dpdmaxdateyear_596T`) → **years before the decision** (suffix `_yrsago`).
  Left as raw years (2017, 2018, ...), they drift upward with application time and act as a hidden time proxy.
- Calendar-month columns (1–12) are dropped: once the year is relative, the month adds no information.

**Aggregations:** numeric → max, mean | dates → max, min | boolean → mean |
string → number of distinct values + the value on the primary record (`num_group1 == 0`) | row count per case.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

import gc
import time

import polars as pl
import polars.selectors as cs
import psutil

from hcrisk.config import FEAT_DIR, check_data
from hcrisk.features import calendar_cols_of, date_cols_of, process_table, to_relative_time
from hcrisk.io import NON_FEATURES, scan, table_exists

check_data()
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB  (close other notebooks' kernels first)")
DATES = scan("base").select("case_id", "date_decision")   # lazy; joined to detail tables for relative time

Available RAM: 7.4 GB  (close other notebooks' kernels first)


## 1. Depth 0: base + static_0 + static_cb_0
One row per case, so a direct left join. Columns with > 95% missing values are dropped here.

In [2]:
t0 = time.time()
lf0 = (scan("base")
       .join(scan("static_0"), on="case_id", how="left")
       .join(scan("static_cb_0"), on="case_id", how="left"))
ycols, mcols = calendar_cols_of(lf0)
lf0 = to_relative_time(lf0.drop(mcols), date_cols_of(lf0), ycols)
df0 = lf0.collect(engine="streaming").with_columns(cs.float().cast(pl.Float32))

null_rate = df0.null_count() / df0.height
drop = [c for c in df0.columns if c not in NON_FEATURES and null_rate[c][0] > 0.95]
df0 = df0.drop(drop)
df0.write_parquet(FEAT_DIR / "depth0.parquet")
print(f"depth0: {df0.shape}, dropped {len(drop)} columns (>95% missing), "
      f"calendar years converted {len(ycols)}, months dropped {len(mcols)}, "
      f"~{df0.estimated_size('gb'):.2f} GB, {time.time() - t0:.0f}s")
del df0, lf0; _ = gc.collect()

depth0: (1526659, 192), dropped 32 columns (>95% missing), calendar years converted 0, months dropped 0, ~1.17 GB, 13s


## 2. Depth 1 / 2 tables
Processed from small to large. Finished tables are skipped, so the cell can be re-run after an interruption.
A warning is printed if a `case_id` appears in more than one chunk (it does not in the Kaggle data).

In [3]:
TABLES = ["other_1", "deposit_1", "debitcard_1", "credit_bureau_b_1", "credit_bureau_b_2",
          "tax_registry_b_1", "person_2", "person_1", "tax_registry_a_1", "tax_registry_c_1",
          "applprev_1", "applprev_2", "credit_bureau_a_1", "credit_bureau_a_2"]
for t in TABLES:
    if table_exists(t):
        process_table(t, DATES)

other_1              rows=    51,109 cols=  12     0.1s
deposit_1            rows=   105,111 cols=   8     0.2s


debitcard_1          rows=   111,772 cols=  10     0.1s


credit_bureau_b_1    rows=    36,500 cols=  84     0.3s
credit_bureau_b_2    rows=    36,447 cols=   8     0.2s


tax_registry_b_1     rows=   150,732 cols=   8     0.2s


person_2             rows= 1,435,105 cols=  18     1.7s


...

tax_registry_c_1     rows=   482,265 cols=   8     0.5s


applprev_1           rows= 1,221,522 cols=  78     9.7s


applprev_2           rows= 1,221,522 cols=   8     2.1s


credit_bureau_a_1    rows= 1,386,273 cols= 148    20.1s


credit_bureau_a_2    rows= 1,385,288 cols=  34    45.1s


## 3. Summary
Missing rates here are **within each table**. They understate true missingness: after joining to all applications,
every case without records becomes missing. Notebook 03 recomputes them on the full population.

In [4]:
rows = []
for f in sorted(FEAT_DIR.glob("*.parquet")):
    lf = pl.scan_parquet(f)
    n = lf.select(pl.len()).collect().item()
    nr = lf.null_count().collect().row(0)
    rows.append((f.stem, n, len(nr) - 1, sum(1 for v in nr[1:] if v / n <= 0.95), round(f.stat().st_size / 1e6, 1)))
pl.DataFrame(rows, schema=["table", "rows", "n_features", "null<=95%", "file_MB"], orient="row").sort("file_MB", descending=True)

table,rows,n_features,null<=95%,file_MB
str,i64,i64,i64,f64
"""credit_bureau_a_1""",1386273,147,143,214.1
"""depth0""",1526659,191,191,177.9
"""applprev_1""",1221522,77,77,95.1
"""credit_bureau_a_2""",1385288,33,29,40.8
"""person_1""",1526659,66,55,39.3
…,…,…,…,…
"""tax_registry_b_1""",150732,7,7,2.4
"""deposit_1""",105111,7,7,1.5
"""debitcard_1""",111772,9,9,0.9
